# Full Pipeline — Integration Test (scoped to 2 sections)

End-to-end execution of the **compiled** LangGraph flow limited to `TARGET_IDS = {"325", "346"}`.

Exercises the real graph machinery with the new **per-section loop**:

- `add_conditional_edges` + `should_retry_section_or_build` routing (retry per section)
- `add_conditional_edges` + `should_next_section_or_end` routing (advance to next section)
- `operator.add` reducer for `validated_rules` (accumulated across sections)
- Builder called once per section, overwriting the same output file each time

The Reader is run manually to filter `parsed_sections` to the 2 target sections.
A variant graph with entry point at `planner` is compiled so we skip re-running the Reader.

**Pre-requisites:**
1. `OPENAI_API_KEY` set in `.env`
2. Qdrant running with the `openapi_reference` collection indexed
3. `LOG_LEVEL=DEBUG` (or `INFO`) in `.env` to see per-node logs

In [ ]:
# Step 1 — Imports and logging setup
import json, os
import config  # noqa: F401 — side-effect: configures logging
from config import get_logger
from config.paths import TSPEC_DATA_TEST_FILE

from langgraph.graph import StateGraph, END
from graph.state import RuleBankState
from graph.conditions import should_retry_section_or_build, should_next_section_or_end
from nodes.reader    import reader_node
from nodes.planner   import planner_node
from nodes.extractor import extractor_node
from nodes.reflector import reflector_node
from nodes.validator import validator_node
from nodes.builder   import builder_node

# Enable DiagnosticCollector for this notebook run:
# Override the module-level singleton in utils.diagnostic so both validator_node
# (record_iteration) and builder_node (save) share the same DiagnosticCollector.
# Also patch config.settings so the DIAGNOSTIC_MODE guard in validator_node passes.
import config.settings as _settings_mod
import utils.diagnostic as _diag_mod
from utils.diagnostic import DiagnosticCollector
import nodes.validator as _val_mod

_settings_mod.DIAGNOSTIC_MODE = True
diagnostic = DiagnosticCollector()
_diag_mod._diagnostic = diagnostic
_val_mod.DIAGNOSTIC_MODE = True  # patch the already-imported local name in validator

logger = get_logger(__name__)

TARGET_IDS = {"325", "346"}
logger.info(f"Imports OK — targeting sections {sorted(TARGET_IDS)}. DiagnosticCollector enabled.")

In [2]:
# Step 2 — Run Reader manually and scope parsed_sections to TARGET_IDS
# Reader is deterministic (no LLM, no loop) — running it outside the graph is safe.
# We do NOT touch excluded_sections_reader: the non-target sections are simply
# dropped from the scope, so the Builder's metadata reflects the scoped run truthfully
# (sections_total = the 2 targets + the Reader's real symbolic-title exclusions).
reader_result = reader_node({
    "main_doc_path":       TSPEC_DATA_TEST_FILE,
    "auxiliary_doc_paths": [],
})

full_parsed    = reader_result["parsed_sections"]
sections_by_id = {s["section_id"]: s for s in full_parsed}

target_parsed = [sections_by_id[sid] for sid in TARGET_IDS if sid in sections_by_id]
missing_ids   = TARGET_IDS - {s["section_id"] for s in target_parsed}
if missing_ids:
    raise ValueError(f"TARGET_IDS not found in parsed_sections: {missing_ids}")

logger.info(
    f"Reader OK — {len(full_parsed)} parsed by Reader, {len(target_parsed)} kept in scope. "
    f"Original excluded_by_reader = {len(reader_result.get('excluded_sections_reader', []))}."
)

2026-04-15 00:17:11 [INFO] nodes.reader: Reader Node started.
2026-04-15 00:17:11 [INFO] nodes.reader: Reader sections: 697 total = 696 for planner + 1 excluded by reader
2026-04-15 00:17:11 [DEBUG] nodes.reader: Loaded 0 auxiliary document(s).
2026-04-15 00:17:11 [INFO] tools.document_tools: discover_specs: directory mode — found 1 .md files
2026-04-15 00:17:11 [DEBUG] tools.document_tools: discover_specs: returning 1 entries
2026-04-15 00:17:11 [DEBUG] tools.document_tools: Loaded 1 OpenAPI reference file(s) from '/home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_rulesbank/src/config/../../data/references/openapi_reference'.
2026-04-15 00:17:11 [INFO] nodes.reader: Reader Node complete.
2026-04-15 00:17:11 [INFO] __main__: Reader OK — 696 parsed by Reader, 2 kept in scope. Original excluded_by_reader = 1.


In [3]:
# Step 3 — Build a scoped variant of the graph starting at the Planner
# Same nodes, same conditional routing, same reducers — only the entry point changes.
def build_scoped_graph():
    g = StateGraph(RuleBankState)
    g.add_node("planner",   planner_node)
    g.add_node("extractor", extractor_node)
    g.add_node("reflector", reflector_node)
    g.add_node("validator", validator_node)
    g.add_node("builder",   builder_node)

    g.add_edge("planner",   "extractor")
    g.add_edge("extractor", "reflector")
    g.add_edge("reflector", "validator")

    # After Validator: retry current section or go to Builder
    g.add_conditional_edges(
        "validator",
        should_retry_section_or_build,
        {"extractor": "extractor", "builder": "builder"},
    )

    # After Builder: process next section or end the pipeline
    g.add_conditional_edges(
        "builder",
        should_next_section_or_end,
        {"extractor": "extractor", "end": END},
    )

    g.set_entry_point("planner")
    return g.compile()

scoped_graph = build_scoped_graph()
logger.info("Scoped graph compiled — entry point = planner.")

2026-04-15 00:17:11 [INFO] __main__: Scoped graph compiled — entry point = planner.


In [4]:
# Step 4 — Invoke the graph with the pre-populated Reader state
initial_state = {
    **reader_result,
    "parsed_sections":              target_parsed,
    "main_doc_path":                TSPEC_DATA_TEST_FILE,
    "auxiliary_doc_paths":          [],
    "validated_rules":              [],
    "validation_errors":            [],
    "section_reflection":           [],
    "validator_section_reflection": [],
    "current_section_idx":          0,
    "section_iteration_count":      0,
    "messages":                     [],
}

# recursion_limit: Planner (1) + per section: up to MAX_ITERATIONS×3 nodes + Builder (1)
# With 2 sections and MAX_ITERATIONS=3: 1 + 2×(3×3+1) = 1 + 2×10 = 21. Use 60 to be safe.
result = scoped_graph.invoke(initial_state, config={"recursion_limit": 60})

output_path = result.get("final_output_path", "")
assert output_path and os.path.isfile(output_path), (
    f"Pipeline did not produce a valid output file: {output_path}"
)
logger.info(f"Pipeline complete — output: {output_path}")
logger.info(
    f"validated: {len(result.get('validated_rules', []))}, "
    f"final errors: {len(result.get('validation_errors', []))}, "
    f"sections processed: {result.get('current_section_idx', '?')}"
)

2026-04-15 00:17:11 [INFO] nodes.planner: Planner Node started.
2026-04-15 00:17:11 [DEBUG] nodes.planner: Sending 2 section(s) to Planner LLM.
2026-04-15 00:17:17 [INFO] nodes.planner: Planner Node complete — 2 section(s) selected, 0 excluded.
2026-04-15 00:17:17 [DEBUG] nodes.planner: Document summary: This 3GPP document section details REST API operations and their parameters, specifically focusing on HTTP methods and resource attribute modifications relevant for OpenAPI rule extraction.
2026-04-15 00:17:17 [INFO] nodes.extractor: Extractor Node started.
2026-04-15 00:17:17 [INFO] sentence_transformers.SentenceTransformer: Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
2026-04-15 00:17:18 [WARNING] huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6877.00it/s]
BertModel LOAD REPORT fr

In [5]:
# Step 5 — Inspect the output JSON and assert invariants
with open(output_path, encoding="utf-8") as f:
    rules_bank = json.load(f)

meta    = rules_bank["metadata"]
summary = rules_bank["summary"]
rules   = rules_bank["rules"]

# Top-level structure
assert set(rules_bank.keys()) == {"metadata", "summary", "rules"}

# Required metadata fields (iterations_run removed in new architecture)
for field in ("source_document", "generated_at", "model", "total_rules",
              "sections_total", "sections_for_planner",
              "sections_planned", "sections_with_rules",
              "excluded_by_reader_count", "excluded_by_planner_count",
              "excluded_by_reader", "excluded_by_planner"):
    assert field in meta, f"Missing metadata field: {field}"

# Count invariants
assert meta["total_rules"] == len(rules)
assert summary["rules_validated_count"] + summary["rules_error_count"] == len(rules)
assert sum(summary["rules_by_type"].values())    == len(rules)
assert sum(summary["rules_by_section"].values()) == len(rules)

# Both sections should have been processed
sections_in_rules = {r["section_id"] for r in rules}
logger.info(f"Sections with rules: {sorted(sections_in_rules)}")
assert len(sections_in_rules) >= 1, "Expected rules from at least 1 section"

# File was written by Builder on each section — verify it reflects the final state
assert meta["total_rules"] == len(rules), "File must reflect all accumulated rules"

logger.info("Output JSON structure OK.")
logger.info("=== METADATA ===")
for k, v in meta.items():
    if not isinstance(v, list):
        logger.info(f"  {k}: {v}")
logger.info(f"  excluded_by_reader  : {meta['excluded_by_reader_count']} section(s)")
logger.info(f"  excluded_by_planner : {meta['excluded_by_planner_count']} section(s)")
logger.info("=== RULES (by section) ===")
for section_id in sorted(sections_in_rules):
    section_rules = [r for r in rules if r["section_id"] == section_id]
    logger.info(f"  [{section_id}] — {len(section_rules)} rule(s)")
    for rule in section_rules:
        logger.info(f"    {rule['rule_type']} / {rule['source_name']} — val_passed={rule.get('validation_passed')}")
        logger.info(f"      {rule['rule_text'][:100]}")

2026-04-15 00:20:18 [INFO] __main__: Sections with rules: ['346']
2026-04-15 00:20:18 [INFO] __main__: Output JSON structure OK.
2026-04-15 00:20:18 [INFO] __main__: === METADATA ===
2026-04-15 00:20:18 [INFO] __main__:   source_document: /home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_rulesbank/src/config/../../../../Dataset/TSpec-LLM/3GPP-clean/Rel-18/28_series/28532-i00.md
2026-04-15 00:20:18 [INFO] __main__:   generated_at: 2026-04-15T00:20:18.881760-03:00
2026-04-15 00:20:18 [INFO] __main__:   model: gpt-4.1-mini
2026-04-15 00:20:18 [INFO] __main__:   total_rules: 7
2026-04-15 00:20:18 [INFO] __main__:   sections_total: 3
2026-04-15 00:20:18 [INFO] __main__:   sections_for_planner: 2
2026-04-15 00:20:18 [INFO] __main__:   sections_planned: 2
2026-04-15 00:20:18 [INFO] __main__:   sections_with_rules: 1
2026-04-15 00:20:18 [INFO] __main__:   excluded_by_reader_count: 1
2026-04-15 00:20:18 [INFO] __m

In [6]:
# Step 6 — Verify per-section write behavior
# The Builder rewrites the same file after each section. The final file must:
#   1. Contain rules from BOTH sections (not just the last one written).
#   2. Have total_rules == len(rules) (metadata updated on every write).
#   3. Have sections_with_rules reflecting all sections that produced rules.

n_sections_processed = result.get("current_section_idx", 0)
logger.info(f"Sections processed by pipeline: {n_sections_processed} / {len(target_parsed)}")
assert n_sections_processed == len(target_parsed), (
    f"Expected {len(target_parsed)} sections processed, got {n_sections_processed}"
)

# All validated rules accumulated via operator.add across sections
total_in_state = len(result.get("validated_rules", []))
assert total_in_state == len(rules), (
    f"State has {total_in_state} validated rules but file has {len(rules)}"
)

# summary.rules_by_section must reflect each section independently
by_section = summary["rules_by_section"]
logger.info(f"rules_by_section: {by_section}")
for section_id in sections_in_rules:
    assert section_id in by_section, f"Section {section_id} missing from rules_by_section"
    count_in_file    = sum(1 for r in rules if r["section_id"] == section_id)
    assert by_section[section_id] == count_in_file, (
        f"[{section_id}] rules_by_section={by_section[section_id]} but file has {count_in_file} rules"
    )

logger.info("Per-section write verification OK — file correctly accumulated all sections.")

2026-04-15 00:20:18 [INFO] __main__: Sections processed by pipeline: 2 / 2
2026-04-15 00:20:18 [INFO] __main__: rules_by_section: {'346': 7}
2026-04-15 00:20:18 [INFO] __main__: Per-section write verification OK — file correctly accumulated all sections.


In [7]:
# Step 7 — Inspect the diagnostic report
from config.paths import OUTPUTS_DIAGNOSTIC_DIR
import glob

doc_stem = os.path.splitext(os.path.basename(TSPEC_DATA_TEST_FILE))[0]
candidates = sorted(glob.glob(os.path.join(OUTPUTS_DIAGNOSTIC_DIR, f"diagnostic_{doc_stem}_*.json")))
assert candidates, f"No diagnostic file found for {doc_stem} in {OUTPUTS_DIAGNOSTIC_DIR}"
diag_path = candidates[-1]
logger.info(f"Diagnostic report: {diag_path}")

with open(diag_path, encoding="utf-8") as f:
    diag = json.load(f)

assert "iterations" in diag
for it in diag["iterations"]:
    logger.info(
        f"\n=== Section attempt {it['iteration']} — "
        f"{it['total_reflected']} reflected | {it['total_validated']} validated | "
        f"{it['total_errors']} errors | error_rate={it['error_rate']:.1%} ==="
    )
    for rule in it["rules"]:
        v = rule["validator"]
        r = rule["reflector"]
        status = "PASS" if v["passed"] else f"FAIL [{v['error_type']}/{v['stage']}]"
        logger.info(f"  {status}  [{rule['section_id']}] {rule['rule_type']} / {rule['source_name']}")
        if not v["passed"]:
            logger.info(f"    → {v['instruction']}")
        if r["flagged"] or r["split_suggestion"] or r["discard_suggestion"]:
            logger.info(
                f"    reflector: conf={r['confidence']:.2f}  flagged={r['flagged']}  "
                f"split={r['split_suggestion']!r}  discard={r['discard_suggestion']}"
            )
    if it["section_reflection"]:
        logger.info(f"  validator_section_reflection: {it['section_reflection']}")

2026-04-15 00:20:18 [INFO] __main__: Diagnostic report: /home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_rulesbank/src/config/../../data/outputs/diagnostic/diagnostic_28532-i00_20260415_002018.json
2026-04-15 00:20:18 [INFO] __main__: 
=== Section attempt 1 — 7 reflected | 5 validated | 2 errors | error_rate=28.6% ===
2026-04-15 00:20:18 [INFO] __main__:   PASS  [346] path_operation / PUT
2026-04-15 00:20:18 [INFO] __main__:     reflector: conf=0.90  flagged=False  split='Optionally split into multiple rules: one for supported query parameters (none), one for request body data structure (Resource), and one for response codes and response body schemas (200, 201, 204, 4xx/5xx).'  discard=False
2026-04-15 00:20:18 [INFO] __main__:   FAIL [discard/semantic]  [346] query_parameter / n/a
2026-04-15 00:20:18 [INFO] __main__:     → Discard — absence of query parameters is not a valid OpenAPI rule. Instead, simpl